# Player career dataframe

One row per player, aggregated across all seasons and categories.

## Columns built here

| Column | Source | Notes |
|---|---|---|
| `player_id` | players.csv | Stable RFFM id |
| `player_name` | players.csv | |
| `birth_year` | players.csv | |
| `seasons_active` | player_season_stats | Sorted list of seasons the player appeared in |
| `seasons_count` | — | len of above |
| `categories` | player_competition_participation | Sorted unique list: PREBENJAMIN, BENJAMIN, … |
| `divisions` | player_competition_participation + competitions | Sorted unique division_level values |
| `clubs` | player_competition_participation | Sorted unique club_name_raw values |
| `teams` | player_competition_participation | Sorted unique team names |
| `competitions` | player_competition_participation | Sorted unique competition names |
| `matches_played` | player_season_stats (sum) | Site-reported total appearances |
| `starter_appearances` | player_season_stats (sum) | |
| `substitute_appearances` | player_season_stats (sum) | |
| `called_up` | player_season_stats (sum) | Called to squad (incl. unused subs) |
| `goals_total` | player_season_stats (sum) | Site-reported; cross-checked below |
| `goals_from_acta` | match_goals (count) | Count from acta enrichment (categories with acta only) |
| `yellow_cards` | player_season_stats (sum) | |
| `red_cards` | player_season_stats (sum) | |
| `second_yellow_cards` | player_season_stats (sum) | |
| `is_goalkeeper_ever` | player_season_stats (any) | True if ever played as GK |
| `captain_appearances` | match_lineups (count) | Appearances as captain (acta categories only) |
| `wins` | match_lineups + matches | Finished matches where player's team won |
| `draws` | match_lineups + matches | |
| `losses` | match_lineups + matches | |
| `win_rate` | — | wins / (wins+draws+losses), NaN if 0 matches |

## Coverage notes

- `player_season_stats` / `player_competition_participation`: available for all seasons 2018-2026, all categories.
- `match_lineups` / `match_goals` / `match_cards` (acta enrichment): all seasons 2018-2026, all categories.
- Win/loss and captain stats come from acta — if a category has no acta coverage they are 0 (check `coverage_manifest.csv`).
- A player can appear in multiple seasons and categories — all rows are aggregated here.

In [1]:
from pathlib import Path
import glob
import pandas as pd
import numpy as np

BASE = Path("../output/processed/rffm")
SEASONS = sorted(d.name for d in BASE.iterdir() if d.is_dir() and d.name[0].isdigit())
print("Seasons:", SEASONS)

Seasons: ['2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']


## 1. Identity — players.csv across all seasons

Same player_id can appear in multiple seasons. We take the most recent scraped row per player_id.

In [2]:
players_frames = []
for s in SEASONS:
    p = BASE / s / "players.csv"
    if p.exists():
        df = pd.read_csv(p, dtype=str, usecols=["player_id", "player_name", "birth_year"])
        df["_season"] = s
        players_frames.append(df)

players_all = pd.concat(players_frames, ignore_index=True)
# Keep latest row per player_id (seasons sorted ascending → last is most recent)
players = (
    players_all
    .sort_values("_season")
    .drop_duplicates(subset="player_id", keep="last")
    [["player_id", "player_name", "birth_year"]]
    .reset_index(drop=True)
)
print(f"Unique players across all seasons: {len(players):,}")
players.head(3)

Unique players across all seasons: 289,346


,player_id,player_name,birth_year
0,10000418,"VINUEZA ALVAREZ, ADRIAN",2003
1,6407172,"OLARU RIBERA, SAMUEL",2010
2,640711,"QUINTERO CASTELLANOS, SERGIO",2002


## 2. Season stats — player_season_stats.csv

Sum numeric stats across all seasons per player. Also collect `seasons_active` list.

In [3]:
STAT_COLS = [
    "called_up", "starter_appearances", "substitute_appearances",
    "matches_played", "goals_total", "yellow_cards", "red_cards",
    "second_yellow_cards",
]

pss_frames = []
for s in SEASONS:
    p = BASE / s / "player_season_stats.csv"
    if p.exists():
        df = pd.read_csv(p, dtype=str)
        df["_season"] = s
        pss_frames.append(df)

pss_all = pd.concat(pss_frames, ignore_index=True)
print(f"Total player_season_stats rows: {len(pss_all):,}")

# Numeric cast
for c in STAT_COLS:
    pss_all[c] = pd.to_numeric(pss_all[c], errors="coerce").fillna(0).astype(int)
pss_all["is_goalkeeper"] = pss_all["is_goalkeeper"].str.lower().eq("true")

# Seasons list per player
seasons_active = (
    pss_all.groupby("player_id")["_season"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .rename("seasons_active")
)

# Sum numeric stats
pss_agg = pss_all.groupby("player_id").agg(
    **{c: (c, "sum") for c in STAT_COLS},
    is_goalkeeper_ever=("is_goalkeeper", "any"),
).join(seasons_active)

pss_agg["seasons_count"] = pss_agg["seasons_active"].str.len()

print(f"Players with season stats: {len(pss_agg):,}")
pss_agg.head(3)

Total player_season_stats rows: 908,400
Players with season stats: 289,346


,called_up,starter_appearances,substitute_appearances,matches_played,goals_total,yellow_cards,red_cards,second_yellow_cards,is_goalkeeper_ever,seasons_active,seasons_count
player_id,,,,,,,,,,,
10000023,160,102,58,138,0,2,0,0,True,"[2019-2020, 2020-2021, 2021-2022, 2022-2023, 2...",7
10000038,21,17,4,21,0,0,0,1,False,"[2020-2021, 2022-2023]",2
10000070,23,23,0,23,0,0,0,0,False,"[2018-2019, 2021-2022]",2


In [6]:
pss_agg.sort_values("goals_total", ascending=False).head(10)

,called_up,starter_appearances,substitute_appearances,matches_played,goals_total,yellow_cards,red_cards,second_yellow_cards,is_goalkeeper_ever,seasons_active,seasons_count
player_id,,,,,,,,,,,
751366,286,252,34,284,395,22,0,1,False,"[2018-2019, 2019-2020, 2020-2021, 2021-2022, 2...",8
2468893,195,154,41,195,335,7,0,0,True,"[2018-2019, 2019-2020, 2020-2021, 2021-2022, 2...",7
4395700,253,213,40,253,316,1,0,0,False,"[2018-2019, 2019-2020, 2020-2021, 2021-2022, 2...",8
11825665,135,121,14,135,312,1,0,0,False,"[2021-2022, 2022-2023, 2023-2024, 2024-2025, 2...",5
10337792,155,146,9,155,291,5,0,1,False,"[2020-2021, 2021-2022, 2022-2023, 2023-2024, 2...",6
13297186,134,133,1,134,286,5,0,1,False,"[2021-2022, 2022-2023, 2023-2024, 2024-2025, 2...",5
2803539,196,126,70,183,282,11,0,1,True,"[2019-2020, 2020-2021, 2021-2022, 2022-2023, 2...",7
12033126,161,159,2,161,277,3,0,0,False,"[2021-2022, 2022-2023, 2023-2024, 2024-2025, 2...",5
11443283,145,140,5,145,263,4,2,0,True,"[2020-2021, 2021-2022, 2022-2023, 2023-2024, 2...",6


## 3. Participation — teams, clubs, competitions, divisions

player_competition_participation.csv gives team/club/competition per registration.
competitions.csv adds `division_level` and `category_base`.

In [7]:
part_frames = []
for s in SEASONS:
    pp = BASE / s / "player_competition_participation.csv"
    cp = BASE / s / "competitions.csv"
    if not pp.exists():
        continue
    part = pd.read_csv(pp, dtype=str,
        usecols=["player_id", "competition_id", "competition", "team", "club_name_raw"])
    if cp.exists():
        comps = pd.read_csv(cp, dtype=str,
            usecols=["competition_id", "category_base", "division_level"])
        part = part.merge(comps, on="competition_id", how="left")
    else:
        part["category_base"] = None
        part["division_level"] = None
    part_frames.append(part)

part_all = pd.concat(part_frames, ignore_index=True)
print(f"Total participation rows: {len(part_all):,}")

def sorted_unique(series: pd.Series) -> list:
    return sorted({v for v in series.dropna() if str(v).strip() and v != "OTHER"})

part_agg = part_all.groupby("player_id").apply(
    lambda g: pd.Series({
        "categories":   sorted_unique(g["category_base"]),
        "divisions":    sorted_unique(g["division_level"]),
        "clubs":        sorted_unique(g["club_name_raw"]),
        "teams":        sorted_unique(g["team"]),
        "competitions": sorted_unique(g["competition"]),
    })
, include_groups=False)

print(f"Players with participation data: {len(part_agg):,}")
part_agg.head(3)

Total participation rows: 1,247,419
Players with participation data: 289,304


,categories,divisions,clubs,teams,competitions
player_id,,,,,
10000023,"[AFICIONADO, ALEVIN, CADETE, INFANTIL, JUVENIL]","[PREFERENTE, PRIMERA, PRIMERA DIVISION AUTONOM...","[C.D.E. F.P.A. LAS ROZAS, C.F. RAYO MAJADAHOND...","[C.D.E. F.P.A. LAS ROZAS , C.D.E. F.P.A. LAS R...","[PREFERENTE AFICIONADO, PREFERENTE ALEVIN, PRE..."
10000038,"[CADETE, INFANTIL]",[SEGUNDA],"[C.D. ENJOY FOOTBALL, C.D.B. BIS]","[C.D. ENJOY FOOTBALL , C.D.B. BIS ]","[SEGUNDA CADETE, SEGUNDA INFANTIL]"
10000070,[BENJAMIN],[PRIMERA],"[C.D. RUPE SAHAGUN, C.D.E. DEPORTIVO PARLA]","[C.D. RUPE SAHAGUN 'B', C.D.E. DEPORTIVO PARLA...",[PRIMERA BENJAMIN F7]


## 4. Acta stats — wins/draws/losses, goals from acta, captain appearances

Source: `match_lineups/<category>.csv` + `matches.csv` (for result) + `match_goals/<category>.csv`.

Strategy to handle 17M lineup rows efficiently:
1. Load all lineups, keep only `match_id`, `team_id`, `player_id`, `is_starter`, `is_captain`.
2. Load all matches (all seasons), build `match_id → (home_team_id, away_team_id, home_score, away_score)`.
3. Join lineups → match result, compute win/draw/loss per row, then group by player_id.

In [8]:
# 4a. Load all match results across seasons
match_frames = []
for s in SEASONS:
    mp = BASE / s / "matches.csv"
    if mp.exists():
        df = pd.read_csv(mp, dtype=str,
            usecols=["match_id", "home_team_id", "away_team_id",
                     "home_score", "away_score", "status"])
        match_frames.append(df)

matches = pd.concat(match_frames, ignore_index=True)
matches = matches[matches["status"] == "finished"].copy()
matches["home_score"] = pd.to_numeric(matches["home_score"], errors="coerce")
matches["away_score"] = pd.to_numeric(matches["away_score"], errors="coerce")
matches = matches.dropna(subset=["home_score", "away_score"])
print(f"Finished matches: {len(matches):,}")

Finished matches: 632,761


In [9]:
# 4b. Load all match_lineups — keep minimal columns
# Process season by season to cap peak memory
from tqdm.auto import tqdm

lineup_chunks = []
for s in tqdm(SEASONS, desc="lineups"):
    for f in sorted((BASE / s / "match_lineups").glob("*.csv")):
        df = pd.read_csv(f, dtype=str,
            usecols=["match_id", "team_id", "player_id", "is_captain"])
        lineup_chunks.append(df)

lineups = pd.concat(lineup_chunks, ignore_index=True)
lineups["is_captain"] = lineups["is_captain"].str.lower().eq("true")
print(f"Total lineup rows: {len(lineups):,}")

lineups:   0%|          | 0/8 [00:00<?, ?it/s]

Total lineup rows: 17,285,901


In [10]:
# 4c. Join lineups → match result, compute outcome per player per match
# Reshape matches to long: one row per (match_id, team_id, opponent_score, own_score)
home = matches[["match_id", "home_team_id", "home_score", "away_score"]].rename(
    columns={"home_team_id": "team_id", "home_score": "own", "away_score": "opp"})
away = matches[["match_id", "away_team_id", "away_score", "home_score"]].rename(
    columns={"away_team_id": "team_id", "away_score": "own", "home_score": "opp"})
match_team = pd.concat([home, away], ignore_index=True)
match_team["win"]  = (match_team["own"] > match_team["opp"]).astype(int)
match_team["draw"] = (match_team["own"] == match_team["opp"]).astype(int)
match_team["loss"] = (match_team["own"] < match_team["opp"]).astype(int)

# Join lineups → outcomes
lineup_results = lineups.merge(
    match_team[["match_id", "team_id", "win", "draw", "loss"]],
    on=["match_id", "team_id"], how="left"
)
lineup_results[["win", "draw", "loss"]] = lineup_results[["win", "draw", "loss"]].fillna(0).astype(int)

# Aggregate by player
acta_wdl = lineup_results.groupby("player_id").agg(
    wins=("win", "sum"),
    draws=("draw", "sum"),
    losses=("loss", "sum"),
    captain_appearances=("is_captain", "sum"),
)
print(f"Players with acta W/D/L data: {len(acta_wdl):,}")
acta_wdl.head(3)

Players with acta W/D/L data: 289,346


,wins,draws,losses,captain_appearances
player_id,,,,
10000023,88,24,51,3
10000038,0,5,16,0
10000070,0,0,23,0


In [11]:
# 4d. Goals from acta (match_goals)
goal_chunks = []
for s in tqdm(SEASONS, desc="goals"):
    for f in sorted((BASE / s / "match_goals").glob("*.csv")):
        df = pd.read_csv(f, dtype=str, usecols=["player_id"])
        goal_chunks.append(df)

if goal_chunks:
    goals_all = pd.concat(goal_chunks, ignore_index=True)
    goals_acta = goals_all["player_id"].value_counts().rename("goals_from_acta").to_frame()
    print(f"Players with acta goal events: {len(goals_acta):,}")
else:
    goals_acta = pd.DataFrame(columns=["goals_from_acta"])
    goals_acta.index.name = "player_id"

goals_acta.head(3)

goals:   0%|          | 0/8 [00:00<?, ?it/s]

Players with acta goal events: 218,820


,goals_from_acta
player_id,
751366,366
11825665,313
4395700,296


## 5. Merge everything into one dataframe

In [12]:
career = (
    players
    .set_index("player_id")
    .join(pss_agg, how="left")
    .join(part_agg, how="left")
    .join(acta_wdl, how="left")
    .join(goals_acta, how="left")
)

# Fill missing numerics
int_cols = STAT_COLS + ["seasons_count", "wins", "draws", "losses", "captain_appearances", "goals_from_acta"]
for c in int_cols:
    if c in career.columns:
        career[c] = career[c].fillna(0).astype(int)

# Fill missing lists
for c in ["seasons_active", "categories", "divisions", "clubs", "teams", "competitions"]:
    career[c] = career[c].apply(lambda v: v if isinstance(v, list) else [])

career["is_goalkeeper_ever"] = career["is_goalkeeper_ever"].fillna(False)

# Win rate (only for players with acta matches)
total_acta = career["wins"] + career["draws"] + career["losses"]
career["win_rate"] = np.where(total_acta > 0, career["wins"] / total_acta, np.nan)

# Reset index so player_id is a column
career = career.reset_index()

print(f"Final dataframe: {len(career):,} players × {len(career.columns)} columns")
print("Columns:", career.columns.tolist())

Final dataframe: 289,346 players × 25 columns
Columns: ['player_id', 'player_name', 'birth_year', 'called_up', 'starter_appearances', 'substitute_appearances', 'matches_played', 'goals_total', 'yellow_cards', 'red_cards', 'second_yellow_cards', 'is_goalkeeper_ever', 'seasons_active', 'seasons_count', 'categories', 'divisions', 'clubs', 'teams', 'competitions', 'wins', 'draws', 'losses', 'captain_appearances', 'goals_from_acta', 'win_rate']


## 6. Quick sanity checks

In [13]:
print("=== Top 10 scorers (site stats) ===")
print(career.nlargest(10, "goals_total")[["player_name", "birth_year", "seasons_count", "matches_played", "goals_total", "categories"]].to_string())

=== Top 10 scorers (site stats) ===
                          player_name birth_year  seasons_count  matches_played  goals_total                                     categories
181452             SANZ ALONSO, DIEGO       2004              8             284          395                  [AFICIONADO, CADETE, JUVENIL]
109276      DEL PINO FERNANDEZ, ERIKA       2007              7             195          335            [ALEVIN, CADETE, INFANTIL, JUVENIL]
208153       VALLADOLID CURIEL, MARIA       2009              8             253          316  [ALEVIN, BENJAMIN, CADETE, INFANTIL, JUVENIL]
276757        BALBOA RODRIGUEZ, ETHAN       2014              5             135          312                [ALEVIN, BENJAMIN, PREBENJAMIN]
229363        CRISTOBAL ALVAREZ, IKER       2013              6             155          291                   [ALEVIN, BENJAMIN, INFANTIL]
286889           BARROSO LOPEZ, PABLO       2015              5             134          286                [ALEVIN, BENJAMI

In [14]:
print("=== Top 10 scorers (acta goals) ===")
print(career.nlargest(10, "goals_from_acta")[["player_name", "birth_year", "seasons_count", "matches_played", "goals_from_acta", "categories"]].to_string())

=== Top 10 scorers (acta goals) ===
                      player_name birth_year  seasons_count  matches_played  goals_from_acta                                     categories
181452         SANZ ALONSO, DIEGO       2004              8             284              366                  [AFICIONADO, CADETE, JUVENIL]
276757    BALBOA RODRIGUEZ, ETHAN       2014              5             135              313                [ALEVIN, BENJAMIN, PREBENJAMIN]
208153   VALLADOLID CURIEL, MARIA       2009              8             253              296  [ALEVIN, BENJAMIN, CADETE, INFANTIL, JUVENIL]
286889       BARROSO LOPEZ, PABLO       2015              5             134              290                [ALEVIN, BENJAMIN, PREBENJAMIN]
276425   SAN JOSE RUIZ, JUAN JOSE       2014              5             161              278                [ALEVIN, BENJAMIN, PREBENJAMIN]
210088     MARTINEZ MARTIN, SARAY       2010              7             183              268    [ALEVIN, CADETE, INFANTIL, J

In [15]:
print("=== Players active in most seasons ===")
print(career.nlargest(10, "seasons_count")[["player_name", "birth_year", "seasons_count", "seasons_active", "clubs"]].to_string())

=== Players active in most seasons ===
                            player_name birth_year  seasons_count                                                                            seasons_active                                                                                                         clubs
139266            QUESADA CEPEDA, DAVID       1996              8  [2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-2023, 2023-2024, 2024-2025, 2025-2026]                 [A.D. ESCUELA DE FUTBOL DE CARABANCHEL, C.D. BETIS SAN ISIDRO, ESCUELA DE FUTBOL CARABANCHEL]
139267       RAMOS MOREL, STARLYN ARIEL       1997              8  [2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-2023, 2023-2024, 2024-2025, 2025-2026]  [A.D. ESCUELA DE FUTBOL USERA, C.D. SAN CRISTOBAL ANGELES, C.D. SAN CRISTOBAL DE LOS ANGELES, VALLECAS C.F.]
139268      ORUE SANABRIA, CARLOS ASIER       1995              8  [2018-2019, 2019-2020, 2020-2021, 2021-2022, 2022-2023, 2023-2024, 2024-2025, 2025-2026]        

In [16]:
print("=== Most captain appearances ===")
print(career.nlargest(10, "captain_appearances")[["player_name", "birth_year", "captain_appearances", "matches_played", "clubs"]].to_string())

=== Most captain appearances ===
                            player_name birth_year  captain_appearances  matches_played                                                              clubs
263907        VAZQUEZ CONTRERAS, SERGIO       1990                  210             228                                                     [CERCEDA C.F.]
253632       LOPEZ BREA BAQUERO, MARCOS       1974                  196             201                                                        [C.D. BOCA]
206116    DEL PUERTO DOMINGUEZ, IGNACIO       2008                  192             204                                      [RAYO CIUDAD ALCOBENDAS C.F.]
176173  LORCA SANCHEZ, CARLOS SEBASTIAN       1990                  179             202  [CLUB ESCUELA DE FUTBOL CONCEPCION, ESCUELA DE FUTBOL CONCEPCION]
286661       LANCHAS HERNANDEZ, ENRIQUE       1986                  179             191                                                     [ARAVACA C.F.]
265500         RODRIGUEZ GINES, ROBER

In [17]:
print("=== Most clubs (transfers) ===")
career["clubs_count"] = career["clubs"].str.len()
print(career.nlargest(10, "clubs_count")[["player_name", "birth_year", "seasons_count", "clubs_count", "clubs"]].to_string())

=== Most clubs (transfers) ===
                          player_name birth_year  seasons_count  clubs_count                                                                                                                                                                                                                                                                                                                   clubs
217391         MARTINEZ MONTES, PEDRO       2005              8           12                                [A.D. SPORTING HORTALEZA, A.D. UNION ADARVE, C.D. GRIÑON, C.D. LEGANES S.A.D., C.D. NUEVO PUERTA BONITA DE CARABANCHEL, C.D. PUERTA BONITA, C.D.A. NAVALCARNERO, CLUB POLID. PARLA ESCUELA, ESCUELA DEPORTIVA MORATALAZ, MOSTOLES C.F., PASILLO VERDE ARGANZUELA, REAL C.D. CARABANCHEL]
253978           GARCIA MARTIN, BRIAN       2009              8           12  [A.D.C. SAN FERMIN, C.D. CANILLAS, C.D. INTER PROMESAS, C.D. LIBERTAD ALCORCON, C.D. OROQUIETA VILLAVERDE BUTARQU

In [18]:
print("=== Win rate (min 50 acta matches, top 10) ===")
mask = (career["wins"] + career["draws"] + career["losses"]) >= 50
print(career[mask].nlargest(10, "win_rate")[["player_name", "birth_year", "wins", "draws", "losses", "win_rate", "clubs"]].to_string())

=== Win rate (min 50 acta matches, top 10) ===
                   player_name birth_year  wins  draws  losses  win_rate                                                                                   clubs
213750     HEREDIA SANZ, ESTER       1989    54      0       1  0.981818  [C.D.B. COSLADA, C.D.E. OLIMPICO DE MADRID, C.D.E. OLIMPICO DE MADRID FUTBOL FEMENINO]
136015  VICENTE SÁNCHEZ, LUCAS       2011    50      1       0  0.980392                                                          [LOS SANTOS DE LA HUMOSA C.F.]
136013  ROSADO BRONCHALO, HUGO       2011    49      1       0  0.980000                                                          [LOS SANTOS DE LA HUMOSA C.F.]
163571     PEREZ GARCIA, CLHOE       2014    49      1       0  0.980000                                                             [C.D. FUENLABRADA ATLANTIS]
166527    MAQUEDA GOMEZ, SARAY       2014    49      1       0  0.980000                                                             [C.D. FUENLABRA

In [19]:
print("=== Coverage: goals_total vs goals_from_acta ===")
# Players who have both — check correlation
both = career[(career["goals_total"] > 0) & (career["goals_from_acta"] > 0)]
print(f"Players with goals in both sources: {len(both):,}")
print(f"Correlation: {both['goals_total'].corr(both['goals_from_acta']):.3f}")
# Large discrepancies (fichajugador covers more categories than acta historically)
diff = both.copy()
diff["diff"] = (diff["goals_total"] - diff["goals_from_acta"]).abs()
print("\nLargest discrepancies (expected — fichajugador aggregates ALL teams, acta scoped per category):")
print(diff.nlargest(5, "diff")[["player_name", "goals_total", "goals_from_acta", "diff", "categories"]].to_string())

=== Coverage: goals_total vs goals_from_acta ===
Players with goals in both sources: 212,485
Correlation: 0.990

Largest discrepancies (expected — fichajugador aggregates ALL teams, acta scoped per category):
                      player_name  goals_total  goals_from_acta  diff                             categories
227467  IZA OLIVEIRA, LAURA AMAIA          151               36   115                      [CADETE, JUVENIL]
223693     JIMENEZ JIMENEZ, PAULA          128               38    90            [CADETE, INFANTIL, JUVENIL]
85155       ALCOVER DORADO, MATEO          137               49    88   [AFICIONADO, JUVENIL, UNIVERSITARIO]
198537       SANCHEZ SALAN, MARIO          239              152    87  [AFICIONADO, CADETE, JUVENIL, SENIOR]
123253     PEREZ GONZALEZ, MONICA          109               25    84       [JUVENIL, SENIOR, UNIVERSITARIO]


## 7. Export to CSV

In [21]:
out = career.copy()

# Serialise list columns as semicolon-separated strings for CSV
for c in ["seasons_active", "categories", "divisions", "clubs", "teams", "competitions"]:
    out[c] = out[c].apply(lambda v: ";".join(v) if isinstance(v, list) else "")

out_path = Path("../output/processed/rffm/player_career.xlsx")
out.to_excel(out_path, index=False)
print(f"Written {len(out):,} rows to {out_path}")

Written 289,346 rows to ..\output\processed\rffm\player_career.xlsx
